# Multi-Model YOLO Validation and Reporting

Run the `run_yolo_validation_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [1]:
# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
else:
    # Running locally
    BASE_DIR = Path.cwd().parent


PROJECT_ROOT = BASE_DIR
SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "yolo_test") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "yolo_test"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Script path: {SCRIPT_PATH}")

# 2. Import validation functions from script
from run_yolo_validation_report import run_validation_pipeline, visualize_predictions
print("✓ Successfully imported validation functions")

# 3. Method Loop over models, run validation, and collect metrics
results_summary = []
validation_results = {}

def test_model(models_configs):
    
    for cfg in models_configs:
        print("=" * 80)
        print(f"Running model: {cfg['name']} | dataset={cfg['dataset']} | split={cfg['split']} | IoU={cfg['iou']}")
        print("=" * 80)
        
        try:
            result = run_validation_pipeline(
                model_name=cfg["name"],
                dataset_name=cfg["dataset"],
                split=cfg["split"],
                iou_threshold=cfg["iou"],
                base_dir=PROJECT_ROOT,
                use_wandb=True,
                save_reports=True,
            )
            
            validation_results[cfg["name"]] = result
            
            overall = result["metrics"]["overall"]
            yolo_overall = result["metrics"]["yolo_metrics"]
            
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "precision_confusion": overall["precision"],
                "recall_confusion": overall["recall"],
                "f1_confusion": overall["f1"],
                "precision_yolo": yolo_overall["precision"],
                "recall_yolo": yolo_overall["recall"],
                "map50": yolo_overall["map50"],
                "map50_95": yolo_overall["map50_95"],
                "params_m": result["model_info"]["params"] / 1e6,
                "size_mb": result["model_info"]["size(MB)"],
                "fps": result["metrics"]["fps"],
                "status": "ok",
                "run_dir": str(result["run_dir"]),
            })
            
        except Exception as e:
            print(f"⚠️ Model {cfg['name']} failed: {e}")
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "status": "error",
            })






Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Torch version: 2.9.1
Device: cpu
Project root: /Users/mahdy/projects/computer_vision_yolo
Script path: /Users/mahdy/projects/computer_vision_yolo/yolo_test/run_yolo_validation_report.py
✓ Successfully imported validation functions


In [2]:
# 4. Select model configurations to test

MODEL_CONFIGS = [
    {"name": "yolov8n", "dataset": "bdd100k_yolo_limited", "split": "test", "iou": 0.5}
]


test_model(MODEL_CONFIGS)


Running model: yolov8n | dataset=bdd100k_yolo_limited | split=test | IoU=0.5
✓ Device: cpu
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov8n_bdd100k_yolo_limited_test_20251123_234646
✓ Dataset loaded
  Total images: 19
  Images with labels: 19
  Label files: 19

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 725
test
✓ Model loaded from /Users/mahdy/projects/computer_vision_yolo/models/yolov8n/yolov8n.pt
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs

📊 Model Information:
  Model: yolov8n
  Classes in model: 80
  Task: detect
  Parameters: 3.2M
  Model Size: 6.2 MB
  FLOPs (640x640): 8.86 GFLOPs
  Model Size: 6.2 MB

Running YOLO validation...
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1 CPU (Apple M3)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.1 ms, read: 285.2±110.3 MB/s, size: 60.5 KB)
val: Scanning /Users/mahdy/projects/computer_vision_yolo/bdd100k_yolo_limited/labels/test.cache... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━

Generating comparisons: 100%|██████████| 6/6 [00:03<00:00,  1.61it/s]


✓ Generated 6 comparison images
  Saved to: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_234646/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_234646/report.pdf
JSON Metrics: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov8n_testing_20251123_234646/metrics_data.json



✓ Weights & Biases run completed successfully


In [10]:
MODEL_CONFIGS = [
{"name": "yolov9s", "dataset": "bdd100k_yolo_limited", "split": "test", "iou": 0.5},
]

test_model(MODEL_CONFIGS)


Running model: yolov9s | dataset=bdd100k_yolo_limited | split=test | IoU=0.5
✓ Device: cpu
✓ W&B logging enabled



✓ Weights & Biases initialized: yolov9s_bdd100k_yolo_limited_test_20251123_235520
✓ Dataset loaded
  Total images: 19
  Images with labels: 19
  Label files: 19

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 725
test
✓ Model loaded from /Users/mahdy/projects/computer_vision_yolo/models/yolov9s/yolov9s.pt
YOLOv9s summary: 544 layers, 7,318,368 parameters, 0 gradients, 27.6 GFLOPs

📊 Model Information:
  Model: yolov9s
  Classes in model: 80
  Task: detect
  Parameters: 7.3M
  Model Size: 14.7 MB
  FLOPs (640x640): 27.56 GFLOPs
  Model Size: 14.7 MB

Running YOLO validation...
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1 CPU (Apple M3)

📊 Model Information:
  Model: yolov9s
  Classes in model: 80
  Task: detect
  Parameters: 7.3M
  Model Size: 14.7 MB
  FLOPs (640x640): 27.56 GFLOPs
  Model Size: 14.7 MB

Running YOLO validation...
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1 CPU (Apple M3)
YOLOv9s summary (fused): 197 layers, 7,198,048 pa

Generating comparisons: 100%|██████████| 6/6 [00:03<00:00,  1.56it/s]



✓ Generated 6 comparison images
  Saved to: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251123_235520/sample_comparisons
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251123_235520/report.pdf
JSON Metrics: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251123_235520/metrics_data.json
✓ COMPREHENSIVE REPORT GENERATED (script)
PDF Report: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251123_235520/report.pdf
JSON Metrics: /Users/mahdy/projects/computer_vision_yolo/yolo_test/runs/yolov9s_testing_20251123_235520/metrics_data.json



✓ Weights & Biases run completed successfully


In [11]:
results_df = pd.DataFrame(results_summary)
results_df

,model_name,dataset,split,iou,precision_confusion,recall_confusion,f1_confusion,precision_yolo,recall_yolo,map50,map50_95,params_m,size_mb,fps,status,run_dir
0,yolov8n,bdd100k_yolo_limited,test,0.5,0.192394,0.573333,0.288107,0.238945,0.149716,0.096223,0.047832,3.157200,6.246372,0.013475,ok,/Users/mahdy/projects/computer_vision_yolo/yol...
1,yolov9s,bdd100k_yolo_limited,test,0.5,0.246085,0.614525,0.351438,0.550625,0.153983,0.097097,0.060985,7.318368,14.674522,0.005523,ok,/Users/mahdy/projects/computer_vision_yolo/yol...


In [13]:
# 5. Compare results across models (tables and plots)
if not results_df.empty and len(results_df[results_df["status"] == "ok"]) > 0:
    
    success_df = results_df[results_df["status"] == "ok"]
    
    sns.set_style("whitegrid")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=100)
    
    # mAP@0.5
    axes[0, 0].bar(success_df["model_name"], success_df["map50"], color='#3498db')
    axes[0, 0].set_title("mAP@0.5 per model (YOLO official)", fontweight='bold')
    axes[0, 0].set_ylabel("mAP@0.5")
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # F1 Score
    axes[0, 1].bar(success_df["model_name"], success_df["f1_confusion"], color='#27ae60')
    axes[0, 1].set_title("F1 (from confusion matrix) per model", fontweight='bold')
    axes[0, 1].set_ylabel("F1 score")
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Model Size vs Performance
    axes[1, 0].scatter(success_df["size_mb"], success_df["map50"], s=100, alpha=0.6)
    for idx, row in success_df.iterrows():
        axes[1, 0].annotate(row["model_name"], (row["size_mb"], row["map50"]), 
                           xytext=(5, 5), textcoords='offset points')
    axes[1, 0].set_xlabel("Model Size (MB)")
    axes[1, 0].set_ylabel("mAP@0.5")
    axes[1, 0].set_title("Model Size vs Performance", fontweight='bold')
    
    # FPS Comparison
    axes[1, 1].bar(success_df["model_name"], success_df["fps"], color='#e74c3c')
    axes[1, 1].set_title("Inference Speed (FPS)", fontweight='bold')
    axes[1, 1].set_ylabel("FPS")
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No successful runs to compare.")

<Figure size 1600x1200 with 4 Axes>

In [18]:
# 8. Per-Class Performance Comparison
if validation_results:
    print("=" * 80)
    print("Per-Class Performance Comparison Across Models")
    print("=" * 80)
    
    # Collect per-class data
    per_class_comparison = []
    for model_name, result in validation_results.items():
        df_metrics = result["df_metrics"]
        for _, row in df_metrics.iterrows():
            per_class_comparison.append({
                "model": model_name,
                "class": row["Class"],
                "precision": row["Precision"],
                "recall": row["Recall"],
                "f1": row["F1-Score"],
                "map50": row["mAP@0.5"],
            })
    
    per_class_df = pd.DataFrame(per_class_comparison)
    
    # Get unique classes
    classes = per_class_df["class"].unique()
    
    # Plot comparison for each metric
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    
    for ax, metric in zip(axes.flatten(), ["precision", "recall", "f1", "map50"]):
        pivot_data = per_class_df.pivot(index="class", columns="model", values=metric)
        pivot_data.plot(kind="bar", ax=ax, width=0.8)
        ax.set_title(f"{metric.upper()} by Class", fontweight='bold', fontsize=14)
        ax.set_ylabel(metric.capitalize())
        ax.set_xlabel("Class")
        ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display detailed table
    print("\n📊 Detailed Per-Class Metrics:")
    display(per_class_df.pivot_table(
        index="class",
        columns="model",
        values=["precision", "recall", "f1", "map50"],
        aggfunc="first"
    ).round(4))

Per-Class Performance Comparison Across Models


<Figure size 2000x1400 with 4 Axes>


📊 Detailed Per-Class Metrics:


f1           map50         precision          recall  \
model         yolov8n yolov9s yolov8n yolov9s   yolov8n yolov9s yolov8n   
class                                                                     
bicycle        0.0000  0.0000  0.0000  0.0000    0.0000  0.0000  0.0000   
bus            0.0000  0.0000  0.0000  0.0000    0.0000  0.0000  0.0000   
car            0.4658  0.5535  0.4737  0.5433    0.3346  0.4173  0.7658   
motorcycle     0.0000  0.0000  0.0000  0.0000    0.0000  0.0000  0.0000   
pedestrian     0.0000  0.0000  0.0000  0.0000    0.0000  0.0000  0.0000   
rider          0.0000  0.0000  0.0129  0.0000    0.0000  0.0000  0.0000   
traffic light  0.0000  0.0000  0.0000  0.0000    0.0000  0.0000  0.0000   
traffic sign   0.0238  0.0879  0.0349  0.0599    0.0130  0.0519  0.1429   
train          0.0000  0.0000  0.1421  0.0765    0.0000  0.0000  0.0000   
truck          0.0000  0.0000  0.0100  0.0000    0.0000  0.0000  0.0000   

                       
model         yolov9s  
class                  
bicycle        0.0000  
bus            0.0000  
car            0.8217  
motorcycle     0.0000  
pedestrian     0.0000  
rider          0.0000  
traffic light  0.0000  
traffic sign   0.2857  
train          0.0000  
truck          0.0000

In [ ]:
# 9. Summary Report
print("=" * 80)
print("SUMMARY REPORT")
print("=" * 80)

if not results_df.empty:
    success_df = results_df[results_df["status"] == "ok"]
    
    if not success_df.empty:
        # Best model by different criteria
        best_map50 = success_df.loc[success_df["map50"].idxmax()]
        best_f1 = success_df.loc[success_df["f1_confusion"].idxmax()]
        best_fps = success_df.loc[success_df["fps"].idxmax()]
        smallest = success_df.loc[success_df["size_mb"].idxmin()]
        
        print("\n🏆 Best Model by mAP@0.5:")
        print(f"   {best_map50['model_name']} - mAP@0.5: {best_map50['map50']:.4f}")
        
        print("\n🏆 Best Model by F1 Score:")
        print(f"   {best_f1['model_name']} - F1: {best_f1['f1_confusion']:.4f}")
        
        print("\n⚡ Fastest Model:")
        print(f"   {best_fps['model_name']} - FPS: {best_fps['fps']:.2f}")
        
        print("\n📦 Smallest Model:")
        print(f"   {smallest['model_name']} - Size: {smallest['size_mb']:.1f} MB")
        
        print("\n" + "=" * 80)
        print("Detailed Comparison Table:")
        print("=" * 80)
        display(success_df[[
            "model_name", "map50", "map50_95", "f1_confusion",
            "precision_yolo", "recall_yolo", "params_m", "size_mb", "fps"
        ]].round(4))
    else:
        print("No successful runs to summarize.")
else:
    print("No results to display.")